# Output Parsing
Language models output text. But there are times where you want to get more structured information than just text back

Output parsers are classes that help structure language model responses. There are two main methods an output parser must implement:

- **Get format instructions**: A method which returns a string containing instructions for how the output of a language model should be formatted.
- **Parse**: A method which takes in a string (assumed to be the response from a language model) and parses it into some structure.
- Output Parsing
    - StrOutputParser
    - JsonOutputParser
    - CSV Output Parser
    - Datatime Output Parser [Now not available with Langchain v1]
    - Structured Output Parser (Pydanitc or Json)
```Pydantinc``` Output Parser

In [1]:
!pip install langchain_ollama
!pip install dotenv
!pip install langchain_core
!pip install pydantic

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)


In [29]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_core.prompts import (
                                        PromptTemplate, 
                                        SystemMessagePromptTemplate, 
                                        HumanMessagePromptTemplate, 
                                        ChatPromptTemplate, 
                                        PromptTemplate
                                    )

load_dotenv()

base_url = "http://localhost:11434"
model_name = 'qwen3:0.6b'

llm = ChatOllama(
    base_url=base_url,
    model = model_name,
)

In [30]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

In [31]:
class Joke(BaseModel):
    """Joke to tell user"""

    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")
    rating: Optional[int] = Field(description="The rating of the joke is from 1 to 10", default=None)

In [32]:
parser = PydanticOutputParser(pydantic_object=Joke)

instruction = parser.get_format_instructions()

print(instruction)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Joke to tell user", "properties": {"setup": {"description": "The setup of the joke", "title": "Setup", "type": "string"}, "punchline": {"description": "The punchline of the joke", "title": "Punchline", "type": "string"}, "rating": {"anyOf": [{"type": "integer"}, {"type": "null"}], "default": null, "description": "The rating of the joke is from 1 to 10", "title": "Rating"}}, "required": ["setup", "punchline"]}
```


In [33]:
prompt = PromptTemplate(
    template='''
    Answer the user query with a joke. Here is your formatting instruction.
    {format_instruction}

    Query: {query}
    Answer:''',
    input_variables=['query'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = prompt | llm

In [34]:
output = chain.invoke({'query':'Tell me a joke about the cat'})

print(output.content)

```json
{
  "description": "Joke to tell user",
  "setup": "Oh, the cat is a clever one!",
  "punchline": "She's the best at hiding her tail!",
  "rating": null
}
```


In [35]:
chain = prompt | llm | parser
output = chain.invoke({'query':'Tell me a joke about the dogs'})
print(output)

setup='In the heart of the woods, there lived a group of dogs.' punchline='But the paws on the ground were too long, and the tail was too long.' rating=None


## Parsing with ```.with_structured_output()``` method
- This method takes a schema as input which specifies the names, types, and descriptions of the desired output attributes.
- The schema can be specified as a TypedDict class, JSON Schema or a Pydantic class.

In [36]:
output = llm.invoke('Tell me a joke about the cat')
print(output.content)

Here's a clever pun on the word "cat":

**"I'm a cat, but I'm not a cat."**

This play on words cleverly uses the word "cat" twice, highlighting the paradox of being both a cat and not. The joke is a classic example of a clever wordplay.


In [37]:
structured_llm = llm.with_structured_output(Joke)

In [38]:
output = structured_llm.invoke('Tell me a joke about the cat')
print(output)

setup='A fluffy, curious cat lounges on a sunbeam, sipping a bowl of water. A curious dog walks by, curious about its world.' punchline='Whisker’s tail twirls in the air as the dog says, “I’m not your pet!” and the cat, now free, curls into the couch with a ball, leaving a pawprint on the floor.' rating=None


## ```JSON``` Output Parser
- Output parsers accept a string or BaseMessage as input and can return an arbitrary type.

In [39]:
from langchain_core.output_parsers import JsonOutputParser

In [40]:
parser = JsonOutputParser(pydantic_object=Joke)
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [41]:
prompt = PromptTemplate(
    template='''
    Answer the user query with a joke. Here is your formatting instruction.
    {format_instruction}

    Query: {query}
    Answer:''',
    input_variables=['query'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = prompt | llm
output = chain.invoke({'query': 'Tell me a joke about the cat'})
print(output.content)

```json
{"setup": "A cat is sitting on a bench. A mouse is jumping up.", "punchline": "But the cat is not a cat."}
```


In [42]:
chain = prompt | llm | parser
output = chain.invoke({'query': 'Tell me a joke about the cat'})
print(output)

{'setup': 'A cat is playing with a ball in a park. The cat is a cat.', 'punchline': 'The cat is a cat. The ball is a ball.', 'rating': 5}


## CSV Output Parser
- This output parser can be used when you want to return a list of comma-separated items.

In [43]:
# value1, values2, values3, so on

from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()

print(parser.get_format_instructions())

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [44]:
format_instruction = parser.get_format_instructions()

prompt = PromptTemplate(
    template='''
    Answer the user query with a list of values. Here is your formatting instruction.
    {format_instruction}

    Query: {query}
    Answer:''',
    input_variables=['query'],
    partial_variables={'format_instruction': format_instruction}
) 

In [45]:
chain = prompt | llm | parser

output = chain.invoke({'query': 'generate my website seo keywords. I have content about the NLP and LLM.'})
print(output)

['NLP', 'LLM', 'AI for NLP', 'machine learning for NLP', 'LLM research', 'NLP applications', 'Large Language Models.']
